# Autonomous Code Writer & Debugging Assistant
This notebook implements an agentic coding workflow. Given a prompt, the assistant writes Python code and unit tests, executes the tests, checks the code structure, automatically fixes code bugs based on tracebacks, and pauses for human verification before writing the finalized file to disk.

### Workflow Architecture
1. **Generate**: The assistant writes the requested Python function and unit tests.
2. **Parallel Verification**:
   - **Test Runner Node**: Executes the unit tests in a safe runtime environment, catching tracebacks.
   - **Syntax & Code Quality Node**: Inspects the code for structure and docstrings.
3. **Evaluate**: Analyzes results. If unit tests fail, it loops back to the Generator Node up to 3 times with the error traceback to self-correct.
4. **Human Review (Interrupt)**: Pauses execution so a developer can review code and run logs, edit code, and approve.
5. **Save**: Writes the final approved code to a local file.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
print("OpenAI API Key loaded:", "OPENAI_API_KEY" in os.environ)
print("LangSmith tracking active:", os.environ.get("LANGCHAIN_TRACING_V2"))

OpenAI API Key loaded: True
LangSmith tracking active: true


### 1. Define State Schema
 track code draft, unit tests, execution outputs, syntax audits, loop counts, and approval state.

In [2]:
from typing import TypedDict, List, Dict, Any

class CodingState(TypedDict):
    task_description: str
    file_name: str
    code_draft: str
    test_code: str
    test_passed: bool
    test_logs: str
    quality_logs: str
    iterations: int
    approved: bool
    status: str

### 2. Define Generator Node
Generates the code implementation and a standard test suite using LLM. If previous test logs are found, it uses them as feedback to debug.

In [3]:
from mock_llm import get_llm
from langchain_core.prompts import ChatPromptTemplate
import json

llm = get_llm(model="gpt-4o-mini", temperature=0.2)

def generate_code_node(state: CodingState):
    print(f"\n=== Node: Generating/Debugging Code (Iteration {state.get('iterations', 0)}) ===")
    
    # Build feedback prompt if tests failed
    feedback = ""
    if state.get("test_logs") and not state.get("test_passed"):
        feedback = f"\nPrevious tests FAILED with the following error output:\n{state['test_logs']}\nPlease debug and fix the code_draft accordingly."

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert Python coder.
Write a solution for the user prompt. Also, write unit tests checking edge cases.
Return the output strictly as a JSON object with keys 'code' and 'tests'.
Do NOT wrap the code in markdown formatting inside the JSON values, write raw python strings.

Example format:
{{
  \"code\": \"def add(a, b):\\n    return a + b\\n\",
  \"tests\": \"assert add(2, 3) == 5\\nassert add(-1, 1) == 0\\n\"
}}"""),
        ("human", "Task: {task_description}\n{feedback}")
    ])
    
    chain = prompt | llm
    response = chain.invoke({
        "task_description": state["task_description"],
        "feedback": feedback
    })
    
    try:
        res = json.loads(response.content.strip().strip("```json").strip("```"))
        code = res.get("code", "")
        tests = res.get("tests", "")
    except Exception as e:
        print("JSON Parse failed, falling back to regex extraction.")
        code = response.content
        tests = ""
        
    return {
        "code_draft": code,
        "test_code": tests,
        "status": "generated"
    }

--- OpenAI API connection failed (Error code: 429 - {'error': {'message': 'You exceeded your c...). Falling back to Mock LLM ---


### 3. Parallel Check Nodes
run the unit tests (using a safe Python exec capture environment) and check style guidelines in parallel.

In [4]:
import sys
import io

# Parallel Node 1: Test Runner
def run_tests_node(state: CodingState):
    print("=== Node (Parallel): Running Unit Tests ===")
    code = state["code_draft"]
    tests = state["test_code"]
    
    full_program = f"{code}\n\n# --- UNIT TESTS ---\n{tests}"
    
    # Capture stdout and stderr
    old_stdout = sys.stdout
    old_stderr = sys.stderr
    captured_out = io.StringIO()
    captured_err = io.StringIO()
    sys.stdout = captured_out
    sys.stderr = captured_err
    
    passed = True
    try:
        # Execute the code in an isolated local dictionary
        local_env = {}
        exec(full_program, {}, local_env)
    except Exception as e:
        passed = False
        import traceback
        traceback.print_exc(file=captured_err)
    finally:
        sys.stdout = old_stdout
        sys.stderr = old_stderr
        
    test_logs = captured_out.getvalue() + captured_err.getvalue()
    if passed:
        test_logs = "All tests executed successfully.\n" + test_logs
        
    return {
        "test_passed": passed,
        "test_logs": test_logs
    }

# Parallel Node 2: Quality & Style Check
def quality_check_node(state: CodingState):
    print("=== Node (Parallel): Checking Code Quality ===")
    code = state["code_draft"]
    logs = []
    
    # Standard style checks (simple linting rule checks)
    if "def " not in code:
        logs.append("Quality Warning: No functions defined in code.")
    if '"""' not in code and "'''" not in code:
        logs.append("Quality Warning: Code lacks module or function docstrings.")
    if len(code.split("\n")) > 100:
        logs.append("Quality Warning: Function exceeds recommended length of 100 lines.")
        
    quality_log_str = "\n".join(logs) if logs else "Code quality satisfies styling standards."
    return {
        "quality_logs": quality_log_str
    }

### 4. Evaluate quality
Determines whether the code meets validation gates.

In [5]:
def evaluate_quality_node(state: CodingState):
    print("=== Node: Evaluating Verification Outputs ===")
    passed = state["test_passed"]
    iterations = state.get("iterations", 0)
    
    # Update iterations if we need to debug
    next_iterations = iterations if passed else iterations + 1
    
    return {
        "iterations": next_iterations,
        "status": "verified" if passed else "failed_checks"
    }

### 5. Review & Publish Nodes
Reviews code (pauses for developer interrupt) and writes the approved script locally.

In [6]:
def human_review_node(state: CodingState):
    print("=== Node: Human Review Intercept ===")
    # Pass-through node; execution will pause here
    return {}

def save_code_to_disk(state: CodingState):
    print("=== Node: Saving Code to Workspace ===")
    if state["approved"]:
        file_path = f"d:/longGraph/{state['file_name']}"
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(state["code_draft"])
        print(f"\n>>> SUCCESS: Code saved to {file_path} <<<")
        return {"status": "completed"}
    else:
        print("\n>>> CANCELLED: Code discarded or unapproved by user. <<<")
        return {"status": "aborted"}

### 6. Build and Compile Workflow Graph

In [7]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

def route_after_evaluation(state: CodingState):
    if state["test_passed"]:
        print("-> Route: Tests passed. Proceeding to Human Review.")
        return "review_path"
    
    if state["iterations"] >= 3:
        print("-> Route: Max debug attempts reached. Escalate to Human Review.")
        return "review_path"
        
    print(f"-> Route: Tests failed. Sending back to Generator Node (Attempt {state['iterations']}/3)")
    return "debug_path"

builder = StateGraph(CodingState)

# Add Nodes
builder.add_node("generate_code", generate_code_node)
builder.add_node("run_tests", run_tests_node)
builder.add_node("quality_check", quality_check_node)
builder.add_node("evaluate", evaluate_quality_node)
builder.add_node("human_review", human_review_node)
builder.add_node("save_code", save_code_to_disk)

# Connections
builder.add_edge(START, "generate_code")

# Fork parallel checks
builder.add_edge("generate_code", "run_tests")
builder.add_edge("generate_code", "quality_check")

# Join parallel checks into evaluator
builder.add_edge("run_tests", "evaluate")
builder.add_edge("quality_check", "evaluate")

# Conditional loop-back for error debugging
builder.add_conditional_edges(
    "evaluate",
    route_after_evaluation,
    {
        "review_path": "human_review",
        "debug_path": "generate_code"
    }
)

builder.add_edge("human_review", "save_code")
builder.add_edge("save_code", END)

# Setup Memory checkpointer & interrupt criteria
memory = MemorySaver()
code_workflow = builder.compile(checkpointer=memory, interrupt_before=["human_review"])

### 7. Run Workflow: Try generating a bug-prone requirement
Let's ask it to implement a function with edge cases (like a custom division function that handles zero, or a prime factors generator).

In [8]:
config = {"configurable": {"thread_id": "debugger-session-1"}}

input_data = {
    "task_description": "Write a function named 'safe_divide(num, denom)' that divides num by denom. If denom is zero, raise a ValueError with message 'Cannot divide by zero'. Write tests checking float division, positive results, and zero division.",
    "file_name": "math_utility.py",
    "code_draft": "",
    "test_code": "",
    "test_passed": False,
    "test_logs": "",
    "quality_logs": "",
    "iterations": 0,
    "approved": False,
    "status": "pending"
}

# Start execution
code_workflow.invoke(input_data, config)


=== Node: Generating/Debugging Code (Iteration 0) ===
=== Node (Parallel): Checking Code Quality ===
=== Node (Parallel): Running Unit Tests ===
=== Node: Evaluating Verification Outputs ===
-> Route: Tests passed. Proceeding to Human Review.


{'task_description': "Write a function named 'safe_divide(num, denom)' that divides num by denom. If denom is zero, raise a ValueError with message 'Cannot divide by zero'. Write tests checking float division, positive results, and zero division.",
 'file_name': 'math_utility.py',
 'code_draft': 'def safe_divide(num, denom):\n    """Divides num by denom. Raises ValueError if denom is zero."""\n    if denom == 0:\n        raise ValueError(\'Cannot divide by zero\')\n    return num / denom\n',
 'test_code': "assert safe_divide(10, 2) == 5.0\nassert safe_divide(4, -1) == -4.0\ntry:\n    safe_divide(1, 0)\n    assert False, 'Expected ValueError'\nexcept ValueError as e:\n    assert str(e) == 'Cannot divide by zero'\n",
 'test_passed': True,
 'test_logs': 'All tests executed successfully.\n',
 'quality_logs': 'Code quality satisfies styling standards.',
 'iterations': 0,
 'approved': False,
 'status': 'verified'}

### 8. Inspect State (Paused for Human Review)
Let's read what the code checker generated and the test outputs.

In [9]:
current_state = code_workflow.get_state(config)

print("Is paused?")
print("Next Node scheduled to run:", current_state.next)

print("\n--- Generated Code ---")
print(current_state.values["code_draft"])

print("\n--- Generated Unit Tests ---")
print(current_state.values["test_code"])

print("\n--- Execution & Unit Test Logs ---")
print(current_state.values["test_logs"])

print("\n--- Quality/Styling Warnings ---")
print(current_state.values["quality_logs"])

Is paused?
Next Node scheduled to run: ('human_review',)

--- Generated Code ---
def safe_divide(num, denom):
    """Divides num by denom. Raises ValueError if denom is zero."""
    if denom == 0:
        raise ValueError('Cannot divide by zero')
    return num / denom


--- Generated Unit Tests ---
assert safe_divide(10, 2) == 5.0
assert safe_divide(4, -1) == -4.0
try:
    safe_divide(1, 0)
    assert False, 'Expected ValueError'
except ValueError as e:
    assert str(e) == 'Cannot divide by zero'


--- Execution & Unit Test Logs ---
All tests executed successfully.


--- Quality/Styling Warnings ---
Code quality satisfies styling standards.


### 9. Update State (Simulate Developer approval and edits)
 review the code, append comments or edits, set `approved = True`, and resume.

In [10]:
revised_code = current_state.values["code_draft"] + "\n# Verified and reviewed by Developer\n"

code_workflow.update_state(
    config,
    {
        "code_draft": revised_code,
        "approved": True
    },
    as_node="human_review"
)
print("Developer approved code and added annotations!")

Developer approved code and added annotations!


### 10. Resume and Finish
 resume the graph execution to write the file to the local directory.

In [11]:
print("Resuming execution...")
final_state = code_workflow.invoke(None, config)
print("\nFinal Workflow Status:", final_state["status"])

Resuming execution...
=== Node: Saving Code to Workspace ===

>>> SUCCESS: Code saved to d:/longGraph/math_utility.py <<<

Final Workflow Status: completed
